In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as sm

In [3]:
# loading data
distances = pd.read_csv('data/distances.csv')
distances = distances[distances['drive_min'] <= 26]

schools = pd.read_csv('data/polling_locations_tagged.csv').drop(columns = [
    'departamento', 'provincia'])
schools = schools.query("distrito == 'Lurin' | distrito == 'Pachacamac' | distrito == 'San Juan de Miraflores'")
treated = schools[schools['treated'] == 1]

tables = pd.read_csv('data/polling_tables.csv').query("Distrito == 'Lurin' | Distrito == 'Pachacamac' | Distrito == 'San Juan de Miraflores'")
tables = tables.drop(columns = ['Departamento', 'Provincia', 'Estado Acta', 'Ubigeo', 'Impugnados'])
control_schools = (tables.merge(distances, how = 'right', left_on = 'Local de Votación', right_on = 'name'))
control_schools['treated'] = 0

treated_schools = tables.merge(treated, how = 'inner', left_on = 'Local de Votación', right_on = 'local_de_votacion')
treated_schools = treated_schools.drop(columns = ['distrito', 'ubigeo', 'local_de_votacion', 'mesas', 'n_mesas'])

In [4]:
candidates_not_needed = ['AHORA NACIÓN - AN - Alfonso López Chau Nava',
       'ALIANZA ELECTORAL VENCEREMOS - Ronald Atencio Sotomayor',
       'ALIANZA PARA EL PROGRESO - César Acuña Peralta',
       'AVANZA PAÍS - PARTIDO DE INTEGRACIÓN SOCIAL - José Daniel Williams Zapata',
       'FE EN EL PERÚ - Álvaro Paz de la Barra',
       'FRENTE POPULAR AGRÍCOLA FÍA DEL PERÚ',
       'FUERZA POPULAR - Keiko Fujimori Higuchi',
       'FUERZA Y LIBERTAD - Giannina Molinelli Aristondo',
       'JUNTOS POR EL PERÚ - Roberto Sánchez Palomino',
       'LIBERTAD POPULAR - Rafael Belaunde Llosa',
       'PARTIDO APRISTA PERUANO - Pitter Valderrama Peña',
       'PARTIDO CIUDADANOS POR EL PERÚ',
       'PARTIDO CÍVICO OBRAS - Ricardo Belmont Cassinelli',
       'PARTIDO DE LOS TRABAJADORES Y EMPRENDEDORES PTE - PERÚ - Napoleón Becerra García',
       'PARTIDO DEMOCRATICO FEDERAL - Armando Massé Fernández',
       'PARTIDO DEMOCRÁTICO SOMOS PERÚ - George Forsyth Sommer',
       'PARTIDO DEMÓCRATA UNIDO PERÚ - Charlie Carrasco Salazar',
       'PARTIDO DEMÓCRATA VERDE - Alex Gonzales Castillo',
       'PARTIDO FRENTE DE LA ESPERANZA 2021 - Fernando Olivera Vega',
       'PARTIDO MORADO - Mesías Guevara Amasifuén',
       'PARTIDO PATRIÓTICO DEL PERÚ - Herbert Caller Gutiérrez',
       'PARTIDO PAÍS PARA TODOS - Carlos Álvarez Loayza',
       'PARTIDO POLÍTICO COOPERACIÓN POPULAR - Yonhy Lescano Ancieta',
       'PARTIDO POLÍTICO INTEGRIDAD DEMOCRÁTICA - Wolfgang Grozo Costa',
       'PARTIDO POLÍTICO NACIONAL PERÚ LIBRE - Vladimir Cerrón Rojas',
       'PARTIDO POLÍTICO PERÚ ACCIÓN - Francisco Diez-Canseco Távara',
       'PARTIDO POLÍTICO PERÚ PRIMERO - Mario Vizcarra Cornejo',
       'PARTIDO POLÍTICO PRIN - Walter Chirinos Purizaga',
       'PARTIDO SICREO - Carlos Espá', 'PERÚ MODERNO - Carlos Jaico Carranza',
       'PODEMOS PERÚ - José Luna Gálvez',
       'PROGRESEMOS - Paul Jaimes Blanco',
       'RENOVACIÓN POPULAR - Rafael López Aliaga Cazorla',
       'SALVEMOS AL PERÚ - Antonio Ortiz Villano',
       'UN CAMINO DIFERENTE - Rosario Fernández Bazán','UNIDAD NACIONAL - Roberto Chiabra León',
       'PRIMERO LA GENTE – COMUNIDAD, ECOLOGÍA, LIBERTAD Y PROGRESO - Marisol Pérez Tello']
treated_schools = treated_schools.drop(columns = candidates_not_needed)
control_schools = control_schools.drop(columns = candidates_not_needed)
treated_schools['stack'] = treated_schools['Local de Votación']

In [8]:
stacked_schools = pd.concat([treated_schools, control_schools])
stacked_schools = stacked_schools.drop(columns = ['name', 'dest_index'])
def votes_prop(df):
    df['votes_prop'] = df['PARTIDO DEL BUEN GOBIERNO - Jorge Nieto Montesinos'] / df['Votos Válidos']
    return df
stacked_schools = stacked_schools.groupby('stack').apply(votes_prop)
stacked_schools = stacked_schools.drop(columns = ['Blancos', 'Nulos', 'Votos Emitidos', 'PARTIDO DEL BUEN GOBIERNO - Jorge Nieto Montesinos', 'VOTOS EN BLANCO', 'VOTOS IMPUGNADOS', 'VOTOS NULOS'])
stacked_schools = stacked_schools.reset_index()

In [14]:
model = sm.ols('votes_prop ~ treated + C(stack)', data = stacked_schools)
model.fit().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             votes_prop   R-squared:                       0.415
Model:                            OLS   Adj. R-squared:                  0.414
Method:                 Least Squares   F-statistic:                     259.2
Date:                Thu, 22 Oct 2026   Prob (F-statistic):               0.00
Time:                        11:13:39   Log-Likelihood:                 8425.8
No. Observations:                4760   AIC:                        -1.682e+04
Df Residuals:                    4746   BIC:                        -1.673e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
===================================================================================================================
                                                      coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------
Intercept                                           0.1370      0.003     42.246      0.000       0.131       0.143
C(stack)[T.IE 6026 VIRGEN DE FATIMA]               -0.0010      0.005     -0.217      0.828      -0.010       0.008
C(stack)[T.IE 6041 ALFONSO UGARTE]                  0.0152      0.003      4.384      0.000       0.008       0.022
C(stack)[T.IE 6151 SAN LUIS GONZAGA]                0.0143      0.003      4.113      0.000       0.007       0.021
C(stack)[T.IE 7078 VIRGEN DE CHAPI]                 0.0025      0.005      0.538      0.591      -0.007       0.012
C(stack)[T.IE 7098 RODRIGO LARA BONILLA]           -0.0006      0.005     -0.129      0.897      -0.010       0.008
C(stack)[T.IE 7104 RAMIRO PRIALE PRIALE]            0.0011      0.005      0.241      0.810      -0.008       0.010
C(stack)[T.IE 7239 SANTISIMO SALVADOR]             -0.0309      0.009     -3.436      0.001      -0.048      -0.013
C(stack)[T.IE 7261 SANTA ROSA DE COLLANAC]         -0.0204      0.004     -4.704      0.000      -0.029      -0.012
C(stack)[T.IE 7267 SEÑOR DE LOS MILAGROS]           0.0040      0.005      0.821      0.412      -0.005       0.013
C(stack)[T.IE PARROQUIAL VIRGEN DE LA ASUNCION]     0.0143      0.003      4.124      0.000       0.008       0.021
C(stack)[T.IE PARROQUIAL VIRGEN DEL ROSARIO]       -0.0251      0.004     -5.797      0.000      -0.034      -0.017
C(stack)[T.IEP EL UNIVERSO DE CESAR VALLEJO]       -0.0197      0.004     -4.498      0.000      -0.028      -0.011
treated                                             0.1778      0.003     54.302      0.000       0.171       0.184
==============================================================================
Omnibus:                      340.201   Durbin-Watson:                   1.057
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              845.685
Skew:                           0.423   Prob(JB):                    2.30e-184
Kurtosis:                       4.883   Cond. No.                         22.1
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Index(['Código Mesa', 'Distrito', 'Local de Votación', 'Electores Hábiles',
       'Votos Emitidos', 'Votos Válidos', 'Blancos', 'Nulos',
       'PARTIDO DEL BUEN GOBIERNO - Jorge Nieto Montesinos', 'VOTOS EN BLANCO',
       'VOTOS IMPUGNADOS', 'VOTOS NULOS', 'treated'],
      dtype='str')